# AMI Analytics - 7.1 Billion Row Queries

**Purpose**: Demonstrate Snowflake's performance on massive time-series data

This notebook showcases:
- Sub-minute queries on 7.1B row AMI table
- Time-series aggregations with clustering optimization
- YoY summer peak comparisons
- Voltage anomaly detection

**Data Scale**:
- 7.1B AMI interval readings (15-minute granularity)
- 596,906 smart meters
- 13 months of data (July 2024 - August 2025)

---

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as F
import pandas as pd

session = get_active_session()
session.use_database("SI_DEMOS")
session.use_schema("PRODUCTION")

# Verify data scale
count = session.sql("SELECT COUNT(*) FROM AMI_INTERVAL_READINGS").collect()[0][0]
print(f"AMI Readings: {count:,} rows")

## 1. Daily Consumption Summary

Aggregate 7.1B rows by day - demonstrates clustering optimization

In [ ]:
%%time
# Daily consumption for August 2025 (summer peak)
daily_usage = session.sql("""
    SELECT 
        DATE_TRUNC('DAY', TIMESTAMP) as DAY,
        COUNT(*) as READINGS,
        COUNT(DISTINCT METER_ID) as ACTIVE_METERS,
        ROUND(SUM(USAGE_KWH), 2) as TOTAL_KWH,
        ROUND(AVG(VOLTAGE), 1) as AVG_VOLTAGE,
        SUM(CASE WHEN VOLTAGE < 114 THEN 1 ELSE 0 END) as LOW_VOLTAGE_EVENTS
    FROM AMI_INTERVAL_READINGS
    WHERE TIMESTAMP >= '2025-08-01' AND TIMESTAMP < '2025-09-01'
    GROUP BY 1
    ORDER BY 1
""").to_pandas()

print(f"Query returned {len(daily_usage)} days")
daily_usage.head(10)

## 2. Peak Hour Analysis

Identify hottest hours with highest demand

In [ ]:
%%time
peak_hours = session.sql("""
    SELECT 
        DATE_TRUNC('HOUR', TIMESTAMP) as HOUR,
        ROUND(SUM(USAGE_KWH), 0) as TOTAL_KWH,
        ROUND(AVG(VOLTAGE), 1) as AVG_VOLTAGE,
        SUM(CASE WHEN VOLTAGE < 114 THEN 1 ELSE 0 END) as VOLTAGE_SAGS
    FROM AMI_INTERVAL_READINGS
    WHERE TIMESTAMP BETWEEN '2025-08-01' AND '2025-08-31'
    GROUP BY 1
    ORDER BY TOTAL_KWH DESC
    LIMIT 20
""").to_pandas()

print("Top 20 Peak Demand Hours (August 2025):")
peak_hours

## 3. Year-over-Year Comparison

Compare July 2024 vs July 2025 summer peaks

In [ ]:
%%time
yoy_comparison = session.sql("""
    WITH july_2024 AS (
        SELECT 
            'July 2024' as PERIOD,
            COUNT(*) as READINGS,
            ROUND(SUM(USAGE_KWH), 0) as TOTAL_KWH,
            ROUND(AVG(USAGE_KWH), 4) as AVG_KWH_PER_READING
        FROM AMI_INTERVAL_READINGS
        WHERE TIMESTAMP >= '2024-07-01' AND TIMESTAMP < '2024-08-01'
    ),
    july_2025 AS (
        SELECT 
            'July 2025' as PERIOD,
            COUNT(*) as READINGS,
            ROUND(SUM(USAGE_KWH), 0) as TOTAL_KWH,
            ROUND(AVG(USAGE_KWH), 4) as AVG_KWH_PER_READING
        FROM AMI_INTERVAL_READINGS
        WHERE TIMESTAMP >= '2025-07-01' AND TIMESTAMP < '2025-08-01'
    )
    SELECT * FROM july_2024
    UNION ALL
    SELECT * FROM july_2025
""").to_pandas()

print("Year-over-Year Summer Peak Comparison:")
yoy_comparison

## 4. Voltage Anomaly Detection

Find meters with repeated voltage issues

In [ ]:
%%time
voltage_issues = session.sql("""
    SELECT 
        METER_ID,
        COUNT(*) as TOTAL_READINGS,
        SUM(CASE WHEN VOLTAGE < 108 THEN 1 ELSE 0 END) as SEVERE_SAGS,
        SUM(CASE WHEN VOLTAGE BETWEEN 108 AND 114 THEN 1 ELSE 0 END) as MODERATE_SAGS,
        SUM(CASE WHEN VOLTAGE > 126 THEN 1 ELSE 0 END) as SWELLS,
        ROUND(100.0 * SUM(CASE WHEN VOLTAGE < 114 THEN 1 ELSE 0 END) / COUNT(*), 2) as SAG_PCT
    FROM AMI_INTERVAL_READINGS
    WHERE TIMESTAMP >= '2025-08-01'
    GROUP BY 1
    HAVING SAG_PCT > 5
    ORDER BY SAG_PCT DESC
    LIMIT 50
""").to_pandas()

print(f"Meters with >5% voltage issues: {len(voltage_issues)}")
voltage_issues.head(20)

## 5. Customer Segment Analysis

Compare consumption patterns across segments

In [ ]:
%%time
segment_analysis = session.sql("""
    SELECT 
        CUSTOMER_SEGMENT_ID,
        COUNT(DISTINCT METER_ID) as METERS,
        COUNT(*) as READINGS,
        ROUND(SUM(USAGE_KWH), 0) as TOTAL_KWH,
        ROUND(AVG(USAGE_KWH), 4) as AVG_KWH_PER_READING,
        ROUND(AVG(VOLTAGE), 1) as AVG_VOLTAGE
    FROM AMI_INTERVAL_READINGS
    WHERE TIMESTAMP >= '2025-08-01' AND TIMESTAMP < '2025-09-01'
    GROUP BY 1
    ORDER BY TOTAL_KWH DESC
""").to_pandas()

print("Consumption by Customer Segment (August 2025):")
segment_analysis

## Key Takeaways

1. **Query Performance**: Sub-minute queries on 7.1B rows with proper clustering
2. **Clustering Strategy**: `CLUSTER BY (DATE_TRUNC('DAY', TIMESTAMP), METER_ID)` optimizes time-series queries
3. **Aggregation Pattern**: Pre-aggregated tables (AMI_MONTHLY_USAGE) for dashboard queries
4. **Anomaly Detection**: SQL-based voltage sag detection at scale